# 1. Implementando um perceptron

Antes de implementar um MLP, preciso implementar um perceptron comum

Vou usar o problema da porta AND

In [5]:
import numpy as np


# Deixando uma seed para os números aleatórios não me lascarem
np.random.seed(42)

class Perceptron:
  def __init__ (self, n_inputs, learning_rate=0.1, n_epochs=10):
      self.weights = np.random.randn(n_inputs)
      self.bias = np.random.randn()
      self.learning_rate = learning_rate
      self.n_epochs = n_epochs

  def decision_function(self, x):
    # Se x >=0, retorna 1, senão 0
    return np.where(x >= 0, 1, 0)

# Predict para rodar depois do treino
  def predict(self, X):
    # multiplica input por peso e adiciona o bias
    # np.dot já multiplica os vetores (produto escalar das matrizes)
    s = np.dot(X, self.weights) + self.bias
    return self.decision_function(s)

  # atualizando os pesos e bias
  def fit(self, X, y):
    for epoch in range(self.n_epochs):
      for xi, target in zip(X, y):
        # Predict de treino
        s = np.dot(xi, self.weights) + self.bias
        pred = int(self.decision_function(s))
        # Calculando o erro
        error = target - pred
        # Ajustando o peso
        self.weights += self.learning_rate * error * xi
        # Ajustando o bias
        self.bias += self.learning_rate * error

X = np.array([[0,0],[0,1],[1,0],[1,1]])
y = np.array([0,0,0,1])

perc = Perceptron(n_inputs=2, learning_rate=0.1, n_epochs=10)
perc.fit(X, y)

print("Predições do Perceptron:", perc.predict(X))
print("Pesos finais:", perc.weights)
print("Bias final:", perc.bias)

Predições do Perceptron: [0 0 0 1]
Pesos finais: [0.19671415 0.0617357 ]
Bias final: -0.2523114618993075


# 2. Implementando um MLP

Vou utilizar o problema XOR

In [6]:
import numpy as np

np.random.seed(42)

# === Funções de ativação ===
# Diferente do perceptron (step), aqui usamos uma ativação derivável,
# porque o backprop precisa da derivada.
def sigmoid(x):
    # 1 / (1 + e^(-x))
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    # derivada da sigmoid. Dica: sig(x) * (1 - sig(x))
    s = sigmoid(x)
    return s * (1 - s)

class MLP:
    def __init__(self, n_inputs, n_hidden, n_outputs, learning_rate=0.1, n_epochs=10000):
        # Camada 1: entrada -> escondida
        self.W1 = np.random.randn(n_inputs, n_hidden) # shape (n_inputs, n_hidden)
        self.b1 = np.random.randn(1, n_hidden)     # shape (1, n_hidden)
        # Camada 2: escondida -> saída
        self.W2 = np.random.randn(n_hidden, n_outputs) # shape (n_hidden, n_outputs)
        self.b2 = np.random.randn(1, n_outputs) # shape (1, n_outputs)

        self.learning_rate = learning_rate
        self.n_epochs = n_epochs

    def forward(self, X):
        # Camada escondida
        self.z1 = np.dot(X, self.W1) + self.b1 # X @ W1 + b1   (combinação linear)
        self.a1 = sigmoid(self.z1)     # ativação(z1)
        # Camada de saída
        self.z2 = np.dot(self.a1, self.W2) + self.b2    # a1 @ W2 + b2
        self.a2 = sigmoid(self.z2)     # ativação(z2)
        return self.a2

    def backward(self, X, y, output):
        # 1) Erro na saída
        erro_saida = output - y         # (output - y)
        # 2) Gradiente na saída (erro * derivada da ativação em z2)
        delta_saida = erro_saida * sigmoid_derivative(self.z2)

        # 3) Propaga o erro para a camada escondida
        erro_oculta = np.dot(delta_saida, self.W2.T)       # delta_saida @ W2.T
        # 4) Gradiente na camada escondida
        delta_oculta = erro_oculta * sigmoid_derivative(self.z1) # erro_oculta * derivada da ativação em z1

        # 5) Atualiza pesos e bias (gradiente descendente)
        #    Dica: dW2 = a1.T @ delta_saida ; db2 = soma de delta_saida
        self.W2 -= self.learning_rate * np.dot(self.a1.T, delta_saida)
        self.b2 -= self.learning_rate * np.sum(delta_saida, axis=0, keepdims=True)
        self.W1 -= self.learning_rate * np.dot(X.T, delta_oculta)
        self.b1 -= self.learning_rate * np.sum(delta_oculta, axis=0, keepdims=True)

    def fit(self, X, y):
        for epoch in range(self.n_epochs):
            # forward em todo o batch
            output = self.forward(X)
            # backward (ajusta pesos)
            self.backward(X, y, output)

            # A cada N épocas, imprimir a loss para acompanhar
            if epoch % 1000 == 0:
                loss = np.mean((output - y) ** 2)
                print(f"Época {epoch:5d} - loss: {loss:.4f}")

    def predict(self, X):
        # roda o forward e aplica o limiar: >= 0.5 -> 1, senão 0
        output = self.forward(X)
        return np.where(output >= 0.5, 1, 0)


# === Problema XOR (não é linearmente separável) ===
X = np.array([[0,0], [0,1], [1,0], [1,1]])
y = np.array([[0], [1], [1], [0]])   # note o shape (4, 1)

mlp = MLP(n_inputs=2, n_hidden=4, n_outputs=1, learning_rate=0.1, n_epochs=10000)
mlp.fit(X, y)

print("Predições do MLP:")
print(mlp.predict(X))

Época     0 - loss: 0.4723
Época  1000 - loss: 0.2155
Época  2000 - loss: 0.0689
Época  3000 - loss: 0.0175
Época  4000 - loss: 0.0083
Época  5000 - loss: 0.0051
Época  6000 - loss: 0.0036
Época  7000 - loss: 0.0028
Época  8000 - loss: 0.0023
Época  9000 - loss: 0.0019
Predições do MLP:
[[0]
 [1]
 [1]
 [0]]
